# SANA Hyperparameter Search and Final Evaluation

Notebook for the full SANA experiment protocol:
- deterministic MS-COCO validation/test split;
- P1 hook-point search on validation;
- P2 alpha refinement on validation;
- final test comparison of best steering vs baseline vs random steering.

The notebook imports all reusable logic from `sana_experiment_utils.py` and uses `per_concept_bank` as the default SANA bank mode.

All COCO data, steering banks, validation sweeps, test runs, and summary tables are saved under Google Drive root `/content/drive/MyDrive/diplom/CASteer/`.
Models are cached under Google Drive in `/content/drive/MyDrive/diplom/CASteer/models/` and reused across Colab sessions.


In [1]:
!git clone https://github.com/irenjzh/CASteer.git

Cloning into 'CASteer'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 77 (delta 29), reused 44 (delta 17), pack-reused 17 (from 1)
Receiving objects: 100% (77/77), 13.56 MiB | 14.24 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [3]:
%cd /content/CASteer/

/content/CASteer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [4]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from sana_experiment_utils import (
    ALPHAS_P1,
    ALPHAS_P2_MASTER,
    DEFAULT_HOOK_POINTS,
    DEFAULT_SANA_BANK_MODE,
    build_refinement_window,
    download_coco_dataset,
    ensure_sana_model_on_disk,
    create_coco_split_manifests,
    get_device,
    load_sana_pipeline,
    load_or_compute_hook_point_banks,
    run_test_evaluation,
    run_validation_sweep,
    smoke_test_setup,
    validate_split_manifests,
)

device = get_device()
print(f'Device: {device}')


Device: cuda


In [6]:
MODEL_ALIAS = 'small'
MODEL_ID = None
BANK_MODE = DEFAULT_SANA_BANK_MODE
DRIVE_ROOT = '/content/drive/MyDrive/diplom/CASteer'
COCO_DIR = os.path.join(DRIVE_ROOT, 'coco')
EXPERIMENT_ROOT = os.path.join(DRIVE_ROOT, 'sana_experiments')
STEERING_BANK_DIR = os.path.join(DRIVE_ROOT, 'steering_vectors_sana')
MODEL_STORAGE_DIR = os.path.join(DRIVE_ROOT, 'models')
SPLIT_SEED = 42
RANDOM_SV_SEED = 12345
IMAGES_PER_PROMPT = 5
VALIDATION_SIZE = 100
TEST_SIZE = 2000
NUM_DENOISING_STEPS = 20
NUM_CONCEPTS = 50
STRATEGY = "all_layers"
HOOK_POINTS_P1 = list(DEFAULT_HOOK_POINTS)
ALPHAS_P1_VALUES = list(ALPHAS_P1)
ALPHAS_P2_VALUES = list(ALPHAS_P2_MASTER)
RUN_SMOKE_TEST = False

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(COCO_DIR).mkdir(parents=True, exist_ok=True)
Path(STEERING_BANK_DIR).mkdir(parents=True, exist_ok=True)
Path(MODEL_STORAGE_DIR).mkdir(parents=True, exist_ok=True)
exp_root = Path(EXPERIMENT_ROOT)
exp_root.mkdir(parents=True, exist_ok=True)
print('Drive root:', Path(DRIVE_ROOT).resolve())
print('COCO dir:', Path(COCO_DIR).resolve())
print('Experiment root:', exp_root.resolve())
print('Steering bank dir:', Path(STEERING_BANK_DIR).resolve())
print('Model storage dir:', Path(MODEL_STORAGE_DIR).resolve())
print('Bank mode:', BANK_MODE)


Experiment root: /content/CASteer/sana_experiments
Bank mode: per_concept_bank


In [7]:
coco_info = download_coco_dataset(COCO_DIR)
coco_info


{'coco_dir': 'coco',
 'annotations_path': 'coco/annotations/captions_val2017.json',
 'val_dir': 'coco/val2017',
 'num_val_images': 5000}

In [8]:
splits = create_coco_split_manifests(
    coco_dir=COCO_DIR,
    output_dir=EXPERIMENT_ROOT,
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    split_seed=SPLIT_SEED,
    seeds=list(range(IMAGES_PER_PROMPT)),
)

split_check = validate_split_manifests(
    validation_manifest=splits['validation_manifest'],
    test_manifest=splits['test_manifest'],
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    images_per_prompt=IMAGES_PER_PROMPT,
)
split_check


{'validation_unique_image_ids': 100,
 'test_unique_image_ids': 2000,
 'images_per_prompt': 5}

In [ ]:
model_info = ensure_sana_model_on_disk(
    model_alias=MODEL_ALIAS,
    model_id=MODEL_ID,
    local_model_root=MODEL_STORAGE_DIR,
)
MODEL_PATH = model_info['local_model_path']
model_info


In [9]:
pipe = load_sana_pipeline(
    model_alias=MODEL_ALIAS,
    model_id=MODEL_ID,
    model_path=MODEL_PATH,
    device=device,
)
print('Loaded model:', MODEL_ALIAS)
print('Local model path:', MODEL_PATH)
print('Transformer blocks:', len(pipe.transformer.transformer_blocks))

if RUN_SMOKE_TEST:
    smoke = smoke_test_setup(model_aliases=(MODEL_ALIAS,), hook_points=HOOK_POINTS_P1, device=device)
    smoke


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded model: small
Transformer blocks: 28


In [ ]:
steering_banks = load_or_compute_hook_point_banks(
    pipe=pipe,
    hook_points=HOOK_POINTS_P1,
    bank_mode=BANK_MODE,
    num_concepts=NUM_CONCEPTS,
    num_denoising_steps=NUM_DENOISING_STEPS,
    bank_dir=STEERING_BANK_DIR,
    device=device,
)
steering_bank_df = pd.DataFrame(
    [
        {
            'hook_point': hook_point,
            'bank_path': payload['bank_path'],
            'num_bank_concepts': len(payload['concept_bank']),
        }
        for hook_point, payload in steering_banks.items()
    ]
)
steering_bank_df


## P1: Hook-point search on validation


In [ ]:
p1_root = exp_root / 'p1_validation_search'
p1_rows, p1_best = run_validation_sweep(
    pipe=pipe,
    validation_manifest=splits['validation_manifest'],
    validation_reference_dir=splits['validation_reference_dir'],
    output_root=str(p1_root),
    model_alias=MODEL_ALIAS,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    hook_points=HOOK_POINTS_P1,
    alphas=ALPHAS_P1_VALUES,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    prepared_banks=steering_banks,
    device=device,
)
p1_df = pd.DataFrame(p1_rows).sort_values(["pareto_rank", "selection_score"], ascending=[True, False])
display(Image(filename=str(p1_root / 'validation_alpha_clip_fid.png')))
p1_df


In [ ]:
best_hook_point = p1_best["hook_point"]
best_alpha_p1 = p1_best["alpha"]
p2_alphas = build_refinement_window(best_alpha_p1, ALPHAS_P2_VALUES, window_size=5)
print('Best hook point from P1:', best_hook_point)
print('Best alpha from P1:', best_alpha_p1)
print('P2 refinement window:', p2_alphas)


## P2: Alpha refinement on validation


In [ ]:
p2_root = exp_root / 'p2_validation_refinement'
p2_rows, p2_best = run_validation_sweep(
    pipe=pipe,
    validation_manifest=splits['validation_manifest'],
    validation_reference_dir=splits['validation_reference_dir'],
    output_root=str(p2_root),
    model_alias=MODEL_ALIAS,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    hook_points=[best_hook_point],
    alphas=p2_alphas,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    prepared_banks=steering_banks,
    device=device,
)
p2_df = pd.DataFrame(p2_rows).sort_values(["pareto_rank", "selection_score"], ascending=[True, False])
display(Image(filename=str(p2_root / 'validation_alpha_clip_fid.png')))
p2_df


In [ ]:
best_hook_point = p2_best["hook_point"]
best_alpha = p2_best["alpha"]
print('Selected best validation config:')
print(p2_best)


## P3: Final test comparison


In [ ]:
test_root = exp_root / 'p3_test_eval'
test_rows = run_test_evaluation(
    pipe=pipe,
    test_manifest=splits['test_manifest'],
    output_root=str(test_root),
    model_alias=MODEL_ALIAS,
    best_hook_point=best_hook_point,
    best_alpha=best_alpha,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    random_sv_seed=RANDOM_SV_SEED,
    device=device,
)
test_df = pd.DataFrame(test_rows)
test_df


In [ ]:
display_cols = [
    'variant', 'hook_point', 'alpha', 'clip_score_mean_mean', 'pick_score_mean_mean',
    'image_reward_mean_mean', 'mps_mean', 'vendi_score_mean'
]
test_df[display_cols].sort_values("variant")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

test_df.plot.bar(x="variant", y="clip_score_mean_mean", ax=axes[0], legend=False, title="CLIP Score")
test_df.plot.bar(x="variant", y="pick_score_mean_mean", ax=axes[1], legend=False, title="PickScore")
test_df.plot.bar(x="variant", y="vendi_score_mean", ax=axes[2], legend=False, title="Vendi")

for ax in axes:
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()


## Optional P4

After P1-P3 are complete, you can add post-hoc experiments for `late_layers`, `early_layers`, or `timestep_scaled` by reusing the same functions with a different `STRATEGY`.
